In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Setup: imports and path constants
#
# All paths are defined here in one place.
# DATASET_1 and DATASET_2 are read-only Kaggle inputs.
# WORK_DIR is /kaggle/working — the only writeable location.
# ═══════════════════════════════════════════════════════════════
import os, glob, json, random
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
from PIL import Image

DATASET_1 = "/kaggle/input/datasets/bennymerryman/rice-detection/rice-hainan.coco"
DATASET_2 = "/kaggle/input/datasets/bennymerryman/rice-detection/rice_detection_for_export.coco"

WORK_DIR        = "/kaggle/working"
TRAIN_JSON      = os.path.join(WORK_DIR, "train_annotations.json")
VAL_JSON        = os.path.join(WORK_DIR, "val_annotations.json")
MODEL_SAVE_PATH = os.path.join(WORK_DIR, "rice_retinanet.pth")

os.makedirs(WORK_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device          : {device}")
print(f"Dataset 1 exists: {os.path.exists(DATASET_1)}")
print(f"Dataset 2 exists: {os.path.exists(DATASET_2)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Inspect: verify images and JSON annotation files
# exist in both datasets before doing any processing.
# Raises AssertionError immediately if anything is missing.
# ═══════════════════════════════════════════════════════════════

def find_images(root):
    return (glob.glob(root + "/**/*.jpg", recursive=True) +
            glob.glob(root + "/**/*.png", recursive=True))

def find_jsons(root):
    return glob.glob(root + "/**/*.json", recursive=True)

imgs1  = find_images(DATASET_1)
imgs2  = find_images(DATASET_2)
jsons1 = find_jsons(DATASET_1)
jsons2 = find_jsons(DATASET_2)

print(f"Dataset 1 — {len(imgs1)} images, {len(jsons1)} JSON file(s)")
if imgs1:  print(f"  Example image : {imgs1[0]}")
if jsons1: print(f"  Example JSON  : {jsons1[0]}")

print(f"Dataset 2 — {len(imgs2)} images, {len(jsons2)} JSON file(s)")
if imgs2:  print(f"  Example image : {imgs2[0]}")
if jsons2: print(f"  Example JSON  : {jsons2[0]}")

assert jsons1, f"No JSON found in {DATASET_1}. Check your Kaggle dataset path."
assert jsons2, f"No JSON found in {DATASET_2}. Check your Kaggle dataset path."
assert imgs1,  f"No images found in {DATASET_1}."
assert imgs2,  f"No images found in {DATASET_2}."

print(f"\nTotal images available: {len(imgs1) + len(imgs2)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load, fix, and merge both COCO datasets
#
# Fixes applied to each dataset:
#   - bbox values cast to float
#   - all category_id set to 1 (rice); 0 is reserved for background
#   - categories list normalised to a single rice entry
#   - file_name rewritten to the absolute path on disk
#   - images whose file cannot be found on disk are dropped
#
# Merge: Dataset 2 image and annotation IDs are offset by the
# maximums from Dataset 1 to prevent ID collisions.
# ═══════════════════════════════════════════════════════════════

RICE_CATEGORY = [{'id': 1, 'name': 'rice', 'supercategory': 'none'}]

def load_and_fix_coco(json_path, image_root):
    with open(json_path) as f:
        data = json.load(f)

    # Build basename -> full path lookup for every image on disk
    on_disk = (glob.glob(image_root + "/**/*.jpg", recursive=True) +
               glob.glob(image_root + "/**/*.png", recursive=True))
    disk_lookup = {os.path.basename(p): p for p in on_disk}

    # Rewrite file_name to absolute path; drop images not found
    valid_images = []
    for img in data['images']:
        basename  = os.path.basename(img['file_name'])
        full_path = disk_lookup.get(basename)
        if full_path:
            img['file_name'] = full_path
            valid_images.append(img)

    valid_ids      = {img['id'] for img in valid_images}
    data['images'] = valid_images

    # Fix annotations: cast bbox to float, collapse category to rice
    valid_anns = []
    for ann in data['annotations']:
        if ann['image_id'] not in valid_ids:
            continue
        ann['bbox']        = [float(v) for v in ann['bbox']]
        ann['category_id'] = 1
        valid_anns.append(ann)

    data['annotations'] = valid_anns
    data['categories']  = RICE_CATEGORY
    return data


print("Loading Dataset 1 ...")
data1 = load_and_fix_coco(jsons1[0], DATASET_1)
print(f"  {len(data1['images'])} images, {len(data1['annotations'])} annotations")

print("Loading Dataset 2 ...")
data2 = load_and_fix_coco(jsons2[0], DATASET_2)
print(f"  {len(data2['images'])} images, {len(data2['annotations'])} annotations")

# Offset Dataset 2 IDs so they do not collide with Dataset 1
max_img_id = max(img['id'] for img in data1['images'])
max_ann_id = max(ann['id'] for ann in data1['annotations'])

for img in data2['images']:
    img['id'] += max_img_id + 1

for ann in data2['annotations']:
    ann['id']       += max_ann_id + 1
    ann['image_id'] += max_img_id + 1

merged = {
    'images'     : data1['images']      + data2['images'],
    'annotations': data1['annotations'] + data2['annotations'],
    'categories' : RICE_CATEGORY,
}

print(f"\nMerged — {len(merged['images'])} images, "
      f"{len(merged['annotations'])} annotations")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Train / validation split  (80 / 20)
#
# Shuffles with a fixed seed for reproducibility.
# Splits images first, then assigns annotations by image_id.
# Both JSON files are written to /kaggle/working/.
# ═══════════════════════════════════════════════════════════════

random.seed(42)
all_images = merged['images'].copy()
random.shuffle(all_images)

split        = int(len(all_images) * 0.8)
train_images = all_images[:split]
val_images   = all_images[split:]

train_ids = {img['id'] for img in train_images}
val_ids   = {img['id'] for img in val_images}

train_anns = [a for a in merged['annotations'] if a['image_id'] in train_ids]
val_anns   = [a for a in merged['annotations'] if a['image_id'] in val_ids]

train_coco = {'images': train_images, 'annotations': train_anns,
              'categories': RICE_CATEGORY}
val_coco   = {'images': val_images,   'annotations': val_anns,
              'categories': RICE_CATEGORY}

with open(TRAIN_JSON, 'w') as f:
    json.dump(train_coco, f)
with open(VAL_JSON, 'w') as f:
    json.dump(val_coco, f)

print(f"Train : {len(train_images):4d} images | {len(train_anns):5d} annotations")
print(f"Val   : {len(val_images):4d} images | {len(val_anns):5d} annotations")
print(f"Saved : {TRAIN_JSON}")
print(f"Saved : {VAL_JSON}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — RiceDataset class and DataLoaders
#
# file_name in the JSON is already an absolute path (set in
# Cell 2), so no extra lookup table is needed.
#
# RetinaNet expects:
#   image  : FloatTensor [C, H, W]  values in [0, 1]
#   target : dict with 'boxes'  FloatTensor [N, 4] in xyxy format
#                  and 'labels' Int64Tensor  [N]
#
# __getitem__ returns None for missing files; collate_fn filters
# those out so they never enter the training loop.
# ═══════════════════════════════════════════════════════════════

class RiceDataset(torch.utils.data.Dataset):
    def __init__(self, json_path):
        with open(json_path) as f:
            coco = json.load(f)
        self.images = coco['images']
        # annotation lookup: image_id -> list of annotation dicts
        self.ann_lookup = {}
        for ann in coco['annotations']:
            self.ann_lookup.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = img_info['file_name']  # absolute path from Cell 2

        if not os.path.exists(img_path):
            return None  # filtered by collate_fn

        img  = Image.open(img_path).convert('RGB')
        img  = T.ToTensor()(img)  # [C,H,W] float32 in [0,1]

        anns = self.ann_lookup.get(img_info['id'], [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w > 1 and h > 1:                    # skip degenerate boxes
                boxes.append([x, y, x + w, y + h]) # xywh -> xyxy
                labels.append(1)                    # 1 = rice

        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,),   dtype=torch.int64)

        return img, {'boxes': boxes_t, 'labels': labels_t}


def collate_fn(batch):
    """Drop None entries (missing images) then unzip images and targets."""
    batch = [b for b in batch if b is not None]
    if not batch:
        return [], []
    return tuple(zip(*batch))


train_dataset = RiceDataset(TRAIN_JSON)
val_dataset   = RiceDataset(VAL_JSON)

train_loader = DataLoader(
    train_dataset, batch_size=2, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    collate_fn=collate_fn, num_workers=2
)

print(f"Train dataset : {len(train_dataset)} images")
print(f"Val dataset   : {len(val_dataset)} images")

# Sanity check — load one sample
sample = train_dataset[0]
if sample is not None:
    img_s, tgt_s = sample
    print(f"Sample image shape : {img_s.shape}")
    print(f"Sample boxes       : {tgt_s['boxes'].shape}")
    print(f"Sample labels      : {tgt_s['labels']}")
else:
    print("WARNING: First sample returned None. Check image paths in Cell 2.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Train RetinaNet
#
# Uses torchvision RetinaNet with pretrained ResNet-50 FPN
# backbone. Detection heads are trained from scratch.
#
# NUM_CLASSES = 2  (0 = background, 1 = rice)
# Optimizer   : SGD  lr=0.001  momentum=0.9  weight_decay=1e-4
# Scheduler   : StepLR  decays lr x0.1 after epoch 9
# Gradient clipping at max_norm=1.0 to prevent exploding gradients.
# Model state dict saved to MODEL_SAVE_PATH after all epochs.
# ═══════════════════════════════════════════════════════════════

from torchvision.models.detection import retinanet_resnet50_fpn

NUM_EPOCHS  = 12
NUM_CLASSES = 2
LR          = 0.001

retinanet = retinanet_resnet50_fpn(
    weights=None,
    weights_backbone='DEFAULT',
    num_classes=NUM_CLASSES
)
retinanet.to(device)

optimizer = torch.optim.SGD(
    retinanet.parameters(), lr=LR,
    momentum=0.9, weight_decay=0.0001
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=9, gamma=0.1
)

print(f"Training on : {device}")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"Batch size  : {train_loader.batch_size}")
print(f"Batches/epoch: {len(train_loader)}")
print()

for epoch in range(1, NUM_EPOCHS + 1):
    retinanet.train()
    total_loss = 0.0
    n_batches  = 0

    for i, (images, targets) in enumerate(train_loader):
        if not images:   # skip empty batches (all files missing)
            continue

        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = retinanet(images, targets)
        loss      = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(retinanet.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

        if i % 20 == 0:
            print(f"  Epoch {epoch:02d}/{NUM_EPOCHS} | "
                  f"batch {i:04d}/{len(train_loader)} | "
                  f"loss {loss.item():.4f}")

    avg = total_loss / max(n_batches, 1)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  avg loss: {avg:.4f}  "
          f"lr: {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()   # once per epoch, after the batch loop

torch.save(retinanet.state_dict(), MODEL_SAVE_PATH)
print(f"\nModel saved -> {MODEL_SAVE_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Visual check: inference on a random validation image
#
# Rebuilds the model from the saved state dict so this cell
# works even after a kernel restart. Shows ground-truth boxes
# (green) alongside predicted boxes (red) side by side.
# Result image is also saved to /kaggle/working/visual_check.png.
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from torchvision.models.detection import retinanet_resnet50_fpn

# ── Reload model from checkpoint ──
NUM_CLASSES = 2
vis_model = retinanet_resnet50_fpn(
    weights=None, weights_backbone=None,
    num_classes=NUM_CLASSES
)
vis_model.load_state_dict(
    torch.load(MODEL_SAVE_PATH, map_location=device)
)
vis_model.to(device)
vis_model.eval()
print(f"Model reloaded from {MODEL_SAVE_PATH}")

# ── Pick a random validation sample ──
random.seed()   # unfixed so you get a different image each run
sample = None
for _ in range(len(val_dataset)):
    idx    = random.randint(0, len(val_dataset) - 1)
    sample = val_dataset[idx]
    if sample is not None:
        break

assert sample is not None, "No valid images found in val_dataset."
img_tensor, target = sample

# ── Run inference ──
with torch.no_grad():
    prediction = vis_model([img_tensor.to(device)])[0]

CONF_THRESHOLD = 0.3
keep   = prediction['scores'] > CONF_THRESHOLD
pred_boxes  = prediction['boxes'][keep].cpu()
pred_scores = prediction['scores'][keep].cpu()

# ── Prepare image for display ──
# ToTensor already returns [0,1] floats; just clamp and convert.
img_np = np.clip(img_tensor.permute(1, 2, 0).numpy(), 0, 1)

# ── Side-by-side plot: ground truth | predictions ──
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, title, box_list, score_list, color in [
    (axes[0], 'Ground truth',
     target['boxes'], None, 'lime'),
    (axes[1], f'Predictions  (threshold {CONF_THRESHOLD})',
     pred_boxes, pred_scores, 'red'),
]:
    ax.imshow(img_np)
    ax.set_title(title, fontsize=12)
    ax.axis('off')
    for j, box in enumerate(box_list):
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        ))
        label_text = (f"rice {score_list[j]:.2f}"
                      if score_list is not None else 'rice (gt)')
        ax.text(x1, y1 - 4, label_text,
                color=color, fontsize=8, backgroundcolor='black')

plt.tight_layout()
save_path = os.path.join(WORK_DIR, 'visual_check.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"GT boxes      : {len(target['boxes'])}")
print(f"Pred boxes    : {len(pred_boxes)}")
print(f"Saved plot to : {save_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Export: confirm all output files before committing
#
# Lists every file written to /kaggle/working/ with its size.
# All files shown here will be committed and available as
# dataset outputs when you click 'Save & Run All (Commit)'.
# ═══════════════════════════════════════════════════════════════

outputs = [
    MODEL_SAVE_PATH,
    TRAIN_JSON,
    VAL_JSON,
    os.path.join(WORK_DIR, 'visual_check.png'),
]

print("Output files in /kaggle/working/:")
all_ok = True
for path in outputs:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  OK      {os.path.basename(path):<35s} {size_mb:7.2f} MB")
    else:
        print(f"  MISSING {path}")
        all_ok = False

print()
if all_ok:
    print("All outputs present. Safe to commit.")
else:
    print("Some outputs are missing. Re-run the cells above before committing.")
